In [1]:
import pandas as pd
import numpy as np
import joblib

df = pd.read_csv("feature_enginn 1.0 dataset.csv")

df.columns = df.columns.str.strip().str.replace(" ", "_")

df.rename(columns={"Magnitue": "Magnitude"}, inplace=True)

print(df.shape)

(3380000, 47)


In [2]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

df.fillna(0, inplace=True)

,flow_duration,Header_Length,Protocol_Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot_size,IAT,Number,Magnitude,Radius,Covariance,Variance,Weight,label
0,0.000000,180.18,16.84,64.00,15.758818,15.758818,0.0,0.0,0.0,0.0,...,0.563055,181.16,8.300745e+07,9.5,19.071659,0.802637,10.738677,0.03,141.55,21
1,0.000000,0.00,1.00,64.00,0.996082,0.996082,0.0,0.0,0.0,0.0,...,0.000000,42.00,8.314936e+07,9.5,9.165151,0.000000,0.000000,0.00,141.55,6
2,0.000000,54.00,6.00,64.00,0.718000,0.718000,0.0,0.0,1.0,0.0,...,0.000000,54.00,8.309409e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,10
3,0.000000,54.00,6.00,64.00,6.211557,6.211557,0.0,0.0,0.0,0.0,...,0.000000,54.00,8.303713e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,13
4,5.003695,114.98,6.11,64.00,0.418744,0.418744,0.0,0.0,0.0,0.0,...,0.026812,53.96,8.333211e+07,9.5,10.391685,0.038221,0.024351,0.03,141.55,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3379995,435.289059,57353.50,14.80,87.80,34.950949,34.950949,0.0,0.0,0.0,0.0,...,54.998151,116.60,1.668484e+08,13.5,14.638305,77.904035,3040.271715,1.00,244.60,26
3379996,0.000000,54.00,6.00,64.00,16.412947,16.412947,0.0,0.0,1.0,0.0,...,0.000000,54.00,8.298546e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,19
3379997,4.421559,133.92,6.00,64.00,0.591195,0.591195,0.0,0.0,1.0,0.0,...,0.000000,54.00,8.336545e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,12
3379998,0.113240,70.20,6.00,64.00,6.355089,6.355089,0.0,0.0,1.0,0.0,...,0.000000,54.00,8.309336e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,10


In [3]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

y_train = train_df['label'].values
y_test = test_df['label'].values

In [4]:
rf_features = [
    'flow_duration','Header_Length','Protocol_Type','Duration',
    'HTTP','HTTPS','DNS','TCP','UDP','ICMP',
    'syn_flag_number','ack_flag_number','rst_flag_number',
    'ack_count','syn_count','rst_count'
]

xgb_features = [
    'Tot_sum','Min','Max','AVG','Std','Tot_size',
    'Radius','Covariance','Variance','Magnitude','Weight'
]

gru_features = [
    'IAT','Rate','Srate','Drate'
]

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=20,
    max_depth=10,
    n_jobs=1,
    random_state=42
)

rf.fit(train_df[rf_features], train_df['label'])

pred = rf.predict(test_df[rf_features])

print("RF:", accuracy_score(test_df['label'], pred))
joblib.dump(rf,"RF_model.pkl")

RF: 0.8426079881656805


['RF_model.pkl']

In [6]:
# Stage 1: RF Gatekeeper - Calculate confidence for multi-class
rf_conf_train = rf.predict_proba(train_df[rf_features])
rf_conf_test = rf.predict_proba(test_df[rf_features])

# Get max confidence for each sample (multi-class handling)
rf_conf_train_max = rf_conf_train.max(axis=1)
rf_conf_test_max = rf_conf_test.max(axis=1)

# Define uncertainty mask: samples where RF is NOT confident
threshold = 0.90
uncertain_mask_train = rf_conf_train_max < threshold
uncertain_mask_test = rf_conf_test_max < threshold

print(f"Training - Confident samples: {(~uncertain_mask_train).sum()}, Uncertain: {uncertain_mask_train.sum()}")
print(f"Test - Confident samples: {(~uncertain_mask_test).sum()}, Uncertain: {uncertain_mask_test.sum()}")

Training - Confident samples: 1023369, Uncertain: 1680631
Test - Confident samples: 255855, Uncertain: 420145


In [7]:
# Stage 2: XGBoost Analyst - Train ONLY on uncertain samples
from xgboost import XGBClassifier

X_xgb_train_uncertain = train_df[xgb_features].iloc[uncertain_mask_train].reset_index(drop=True)
y_train_uncertain = y_train[uncertain_mask_train]

X_xgb_test = test_df[xgb_features]

xgb = XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    n_estimators=20,
    max_depth=5,
    random_state=42,
    n_jobs=1
)

# Train only on uncertain samples
xgb.fit(X_xgb_train_uncertain, y_train_uncertain)

# Get predictions and probabilities on full test set
xgb_pred_test = xgb.predict(X_xgb_test)
xgb_conf_test = xgb.predict_proba(X_xgb_test)
xgb_conf_test_max = xgb_conf_test.max(axis=1)

print(f"XGBoost trained on {len(X_xgb_train_uncertain)} uncertain samples")
print(f"XGBoost classes: {xgb.classes_}")

joblib.dump(xgb, "xgb_model.pkl")

C:\Users\avihs\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [16:12:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost trained on 1680631 uncertain samples
XGBoost classes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33]


['xgb_model.pkl']

In [8]:
from sklearn.preprocessing import StandardScaler

gru_scaler = StandardScaler()

X_gru_train = gru_scaler.fit_transform(train_df[gru_features])
X_gru_test = gru_scaler.transform(test_df[gru_features])

joblib.dump(gru_scaler, "gru_scaler.pkl")

['gru_scaler.pkl']

In [9]:
train_df['cum_time'] = train_df['IAT'].cumsum()
test_df['cum_time'] = test_df['IAT'].cumsum()

bucket_size = 1000

train_df['time_bucket'] = (train_df['cum_time']//bucket_size).astype(int)
test_df['time_bucket'] = (test_df['cum_time']//bucket_size).astype(int)

train_df['session'] = train_df['Protocol_Type'].astype(str) + "_" + train_df['time_bucket'].astype(str)
test_df['session'] = test_df['Protocol_Type'].astype(str) + "_" + test_df['time_bucket'].astype(str)

In [10]:
def create_sequences(df, features, seq_len=10):
    X, y = [], []

    for _, group in df.groupby('session'):
        data = group[features].values
        labels = group['label'].values

        for i in range(len(data)):
            seq = data[max(0, i-seq_len):i+1]

            if len(seq) < seq_len:
                pad = np.zeros((seq_len-len(seq), len(features)))
                seq = np.vstack((pad, seq))

            X.append(seq)
            y.append(labels[i])

    return np.array(X), np.array(y)

X_gru_seq_train, y_gru_seq_train = create_sequences(train_df, gru_features)
X_gru_seq_test, y_gru_seq_test = create_sequences(test_df, gru_features)

In [11]:
# Stage 3: GRU Behavioral Model (Multi-class)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout

num_classes = len(np.unique(y_train))

gru_model = Sequential([
    GRU(64, input_shape=(X_gru_seq_train.shape[1], X_gru_seq_train.shape[2]), return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')  # Multi-class output
])

gru_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # For integer class labels
    metrics=['accuracy']
)

print(f"GRU Model - Training on {len(X_gru_seq_train)} sequences with {num_classes} classes")
gru_model.fit(X_gru_seq_train, y_gru_seq_train, epochs=10, batch_size=32, validation_split=0.2, verbose=0)

# Get predictions and probabilities
gru_pred_test = gru_model.predict(X_gru_seq_test, verbose=0)
gru_pred_test_class = np.argmax(gru_pred_test, axis=1)
gru_conf_test_max = gru_pred_test.max(axis=1)

print(f"GRU Accuracy on test: {np.mean(gru_pred_test_class == y_gru_seq_test):.4f}")

gru_model.save("gru_model.keras")

C:\Users\avihs\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


GRU Model - Training on 2704000 sequences with 34 classes
GRU Accuracy on test: 0.1666


In [12]:
# Stage 4: Cascade Aggregation (RF + XGBoost ONLY - no min_len)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Get RF predictions
rf_pred_test = rf.predict(test_df[rf_features])

# Create cascade decision: if RF confident, use RF; else use XGBoost
final_pred_cascade = np.zeros(len(test_df), dtype=int)

for i in range(len(test_df)):
    if rf_conf_test_max[i] >= threshold:
        # RF is confident - use its prediction
        final_pred_cascade[i] = rf_pred_test[i]
    else:
        # RF is uncertain - use XGBoost prediction
        final_pred_cascade[i] = xgb_pred_test[i]

print("=" * 60)
print("STAGE 4: CASCADED EVALUATION (RF -> XGBoost)")
print("=" * 60)
print(f"\nCascade Statistics:")
print(f"  Samples handled by RF (confident):     {(rf_conf_test_max >= threshold).sum()} ({100*(rf_conf_test_max >= threshold).sum()/len(test_df):.1f}%)")
print(f"  Samples passed to XGBoost (uncertain): {uncertain_mask_test.sum()} ({100*uncertain_mask_test.sum()/len(test_df):.1f}%)")

print(f"\nRF Alone Performance:")
rf_pred_test_all = rf.predict(test_df[rf_features])
print(f"  Accuracy: {accuracy_score(y_test, rf_pred_test_all):.4f}")
print(f"  Precision (macro): {precision_score(y_test, rf_pred_test_all, average='macro'):.4f}")
print(f"  Recall (macro): {recall_score(y_test, rf_pred_test_all, average='macro'):.4f}")
print(f"  F1 (macro): {f1_score(y_test, rf_pred_test_all, average='macro'):.4f}")

print(f"\nXGBoost Alone Performance (on full test set):")
print(f"  Accuracy: {accuracy_score(y_test, xgb_pred_test):.4f}")
print(f"  Precision (macro): {precision_score(y_test, xgb_pred_test, average='macro'):.4f}")
print(f"  Recall (macro): {recall_score(y_test, xgb_pred_test, average='macro'):.4f}")
print(f"  F1 (macro): {f1_score(y_test, xgb_pred_test, average='macro'):.4f}")

print(f"\nCascade (RF -> XGBoost) Performance:")
print(f"  Accuracy: {accuracy_score(y_test, final_pred_cascade):.4f}")
print(f"  Precision (macro): {precision_score(y_test, final_pred_cascade, average='macro'):.4f}")
print(f"  Recall (macro): {recall_score(y_test, final_pred_cascade, average='macro'):.4f}")
print(f"  F1 (macro): {f1_score(y_test, final_pred_cascade, average='macro'):.4f}")

STAGE 4: CASCADED EVALUATION (RF -> XGBoost)

Cascade Statistics:
  Samples handled by RF (confident):     255855 (37.8%)
  Samples passed to XGBoost (uncertain): 420145 (62.2%)

RF Alone Performance:
  Accuracy: 0.8426
  Precision (macro): 0.6065
  Recall (macro): 0.4686
  F1 (macro): 0.4886

XGBoost Alone Performance (on full test set):
  Accuracy: 0.3175
  Precision (macro): 0.2613


C:\Users\avihs\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\avihs\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


  Recall (macro): 0.2135
  F1 (macro): 0.1893

Cascade (RF -> XGBoost) Performance:
  Accuracy: 0.6722
  Precision (macro): 0.4014
  Recall (macro): 0.3317
  F1 (macro): 0.3288


C:\Users\avihs\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
# Stage 5: Logistic Regression Judge (Conflict Resolver) - FAST VERSION
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample

# Identify conflict samples on training set (where RF and XGBoost disagree)
rf_pred_train = rf.predict(train_df[rf_features])
xgb_pred_train_all = xgb.predict(train_df[xgb_features])  # Get XGB predictions on full train set

conflict_mask_train = (rf_pred_train != xgb_pred_train_all)

print("=" * 60)
print("STAGE 5: LOGISTIC REGRESSION CONFLICT RESOLVER")
print("=" * 60)
print(f"\nConflict Analysis (Training Set):")
print(f"  Total conflict samples: {conflict_mask_train.sum()} ({100*conflict_mask_train.sum()/len(train_df):.1f}%)")

if conflict_mask_train.sum() > 0:
    # STEP 1: STRICT CONFLICT FILTER - Extract ONLY conflict samples
    X_svm_train_features = np.hstack([
        train_df[rf_features].values,
        rf_conf_train
    ])
    
    X_svm_train_conflicts = X_svm_train_features[conflict_mask_train]
    y_svm_train_conflicts = y_train[conflict_mask_train]
    
    # STEP 2: PRINT SIZE - CHECK HOW MANY CONFLICTS
    print(f"\nConflict samples for training: {len(X_svm_train_conflicts)}")
    
    # STEP 3: SAMPLE IF TOO BIG
    if len(X_svm_train_conflicts) > 50000:
        print(f"⚠️  Conflict set too large ({len(X_svm_train_conflicts)} > 50k). Resampling to 30k...")
        X_svm_train_conflicts, y_svm_train_conflicts = resample(
            X_svm_train_conflicts,
            y_svm_train_conflicts,
            n_samples=30000,
            random_state=42,
            stratify=y_svm_train_conflicts
        )
        print(f"✅ Resampled to {len(X_svm_train_conflicts)} conflict samples")
    
    # STEP 4 & 5: REPLACE SVM with LOGISTIC REGRESSION and TRAIN
    print(f"\n🚀 Training Logistic Regression on {len(X_svm_train_conflicts)} conflict samples...")
    judge = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
    judge.fit(X_svm_train_conflicts, y_svm_train_conflicts)
    print(f"✅ Training complete!")
    
    # Prepare test set features for conflict resolution
    X_svm_test_features = np.hstack([
        test_df[rf_features].values,
        rf_conf_test
    ])
    
    # Get RF and XGB predictions on test set for conflict detection
    rf_pred_test = rf.predict(test_df[rf_features])
    xgb_pred_test_full = xgb.predict(test_df[xgb_features])
    
    # Identify conflicts on test set
    conflict_mask_test = (rf_pred_test != xgb_pred_test_full)
    
    print(f"\nConflict Analysis (Test Set):")
    print(f"  Conflict samples: {conflict_mask_test.sum()} ({100*conflict_mask_test.sum()/len(test_df):.1f}%)")
    
    # Initialize final predictions with cascade results
    final_pred_with_svm = final_pred_cascade.copy()
    
    # Override conflict cases with Logistic Regression predictions
    if conflict_mask_test.sum() > 0:
        judge_pred_conflicts = judge.predict(X_svm_test_features[conflict_mask_test])
        final_pred_with_svm[conflict_mask_test] = judge_pred_conflicts
    
    print(f"\nFinal System Performance (RF -> XGBoost -> Logistic Regression Judge):")
    print(f"  Accuracy: {accuracy_score(y_test, final_pred_with_svm):.4f}")
    print(f"  Precision (macro): {precision_score(y_test, final_pred_with_svm, average='macro', zero_division=0):.4f}")
    print(f"  Recall (macro): {recall_score(y_test, final_pred_with_svm, average='macro', zero_division=0):.4f}")
    print(f"  F1 (macro): {f1_score(y_test, final_pred_with_svm, average='macro', zero_division=0):.4f}")
    
    print(f"\nDetailed Classification Report (Final System):")
    print(classification_report(y_test, final_pred_with_svm, zero_division=0))
    
    joblib.dump(judge, "svm_model.pkl")
    print("\n✅ Logistic Regression Judge saved as 'svm_model.pkl'")
else:
    print("No conflicts found - Judge not trained.")
    final_pred_with_svm = final_pred_cascade.copy()

print("\n" + "=" * 60)
print("SUMMARY: DECISION PIPELINE")
print("=" * 60)
print(f"Stage 1 - RF Gatekeeper: {(rf_conf_test_max >= threshold).sum()} samples")
print(f"Stage 2 - XGBoost Analyst: {uncertain_mask_test.sum()} samples")
print(f"Stage 5 - Logistic Regression Judge: {conflict_mask_test.sum()} conflict samples (resolved)")
print("=" * 60)


STAGE 5: SVM CONFLICT RESOLVER

Conflict Analysis (Training Set):
  Conflict samples: 1658331 (61.3%)


In [ ]:
# GRU Behavioral Model - Separate Evaluation (Sequence-based)
print("=" * 60)
print("GRU BEHAVIORAL MODEL EVALUATION (Separate Pipeline)")
print("=" * 60)
print(f"\nNote: GRU operates on sequences, not individual samples.")
print(f"GRU Dataset Size: {len(X_gru_seq_test)} sequences")
print(f"Main Pipeline Size: {len(test_df)} samples")
print(f"\nGRU Behavioral Analysis:")
print(f"  Test Accuracy: {accuracy_score(y_gru_seq_test, gru_pred_test_class):.4f}")
print(f"  Precision (macro): {precision_score(y_gru_seq_test, gru_pred_test_class, average='macro'):.4f}")
print(f"  Recall (macro): {recall_score(y_gru_seq_test, gru_pred_test_class, average='macro'):.4f}")
print(f"  F1 (macro): {f1_score(y_gru_seq_test, gru_pred_test_class, average='macro'):.4f}")
print(f"\nGRU Classification Report:")
print(classification_report(y_gru_seq_test, gru_pred_test_class))
print("\nNote: GRU is kept as a separate behavioral analyzer.")
print("Future work: Implement sequence-to-row alignment for full integration.")

GRU BEHAVIORAL MODEL EVALUATION (Separate Pipeline)

Note: GRU operates on sequences, not individual samples.


NameError: name 'X_gru_seq_test' is not defined

In [ ]:
# Model Summary and Artifacts
import os

print("=" * 60)
print("SAVED MODELS AND ARTIFACTS")
print("=" * 60)

model_files = [
    ("RF Model", "RF_model.pkl"),
    ("XGBoost Model", "xgb_model.pkl"),
    ("SVM Model", "svm_model.pkl"),
    ("GRU Model", "gru_model.keras"),
    ("GRU Scaler", "gru_scaler.pkl"),
    ("Label Encoder", "label_encoder.pkl")
]

# Save label encoder
joblib.dump(le, "label_encoder.pkl")

print("\nModel Files:")
for name, filepath in model_files:
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024*1024)
        print(f"  ✓ {name:20s} : {filepath:30s} ({size_mb:.2f} MB)")
    else:
        if name != "SVM Model":  # SVM might not exist if no conflicts
            print(f"  ✗ {name:20s} : {filepath:30s} (NOT FOUND)")

print("\n" + "=" * 60)
print("SYSTEM ARCHITECTURE SUMMARY")
print("=" * 60)
print("""
Stage 1 - Random Forest (Gatekeeper)
  ├─ Input: 16 features (RF-specific)
  ├─ Output: Prediction + Confidence
  └─ Role: Initial confident/uncertain classification

Stage 2 - XGBoost (Analyst)
  ├─ Input: 11 features (XGB-specific) - ONLY for uncertain samples
  ├─ Output: Prediction + Confidence
  └─ Role: Deeper analysis of uncertain cases

Stage 3 - GRU (Behavioral Analyzer) [SEPARATE PIPELINE]
  ├─ Input: 4 features (IAT, Rate, Srate, Drate) - Sequences
  ├─ Output: Temporal pattern predictions
  └─ Role: Sequence-based behavioral analysis

Stage 5 - SVM (Judge)
  ├─ Input: RF features + RF confidence [conflict cases only]
  ├─ Output: Final decision on conflicts
  └─ Role: Resolve disagreements between RF and XGB

Final Cascade: IF (RF_confidence >= 0.90) → USE RF
               ELSE IF (RF_pred == XGB_pred) → USE XGB
               ELSE → USE SVM decision
""")

print("\n" + "=" * 60)
print("NEXT STEPS FOR FULL INTEGRATION")
print("=" * 60)
print("""
1. Implement sequence-to-row alignment for GRU
2. Create aggregation layer combining all three models
3. Validate on additional test datasets
4. Optimize cascade thresholds (currently 0.90)
5. Consider temporal windowing for GRU integration
""")